Copyright (c) Microsoft Corporation. All rights reserved.  
Licensed under the MIT License.

## Retrieve ONNX Model That Outputs Attention Weights

In [ ]:
import os
os.environ["HF_HOME"] = "/scratch/rjloh/hf_cache"

In [ ]:
# onnx_model_path = "gpt2.onnx"
onnx_model_path = "/afs/csail.mit.edu/u/r/rjloh/rachel_run_model/convertedModel.onnx"

In [ ]:
!{sys.executable} -m onnxruntime.transformers.models.gpt2.convert_to_onnx -m $model_name_or_path --output $onnx_model_path -o -p fp32 -t 10 >export_output.txt 2>&1

## ONNX Runtime Inference ##

We can use ONNX Runtime to inference. The inputs are dictionary with name and numpy array as value, and the output is list of numpy array. Note that both input and output are in CPU. When you run the inference in GPU, it will involve data copy between CPU and GPU for input and output.

Let's create an inference session for ONNX Runtime given the exported ONNX model, and see the output.

In [ ]:
# !apt install fluidsynth
# !git clone https://github.com/jthickstun/anticipation.git
# %pip install -U pip setuptools wheel
# %pip install tokenizers --prefer-binary
# %pip install ./anticipation
# %pip install -r anticipation/requirements.txt
# %pip install midi2audio
# %pip install mido

In [ ]:
import sys,time

import midi2audio
import transformers
from transformers import AutoModelForCausalLM
import torch
import matplotlib.pyplot as plt

from IPython.display import Audio

from anticipation import ops
from anticipation.sample import generate, nucleus, safe_logits, future_logits, instr_logits
from anticipation.tokenize import extract_instruments
from anticipation.convert import events_to_midi,midi_to_events
from anticipation.visuals import visualize
from anticipation.config import *
from anticipation.vocab import *
from anticipation.convert import ANTICIPATE, events_to_compound, compound_to_midi


## Convert MIDI File to Input Tokens

In [ ]:
from midi2audio import FluidSynth
from IPython.display import Audio

fs = midi2audio.FluidSynth('/usr/share/sounds/sf2/FluidR3_GM.sf2')

# the MIDI synthesis script
def synthesize_og(fs, tokens):
    mid = events_to_midi(tokens)
    mid.save('tmp.mid')
    fs.midi_to_audio('tmp.mid', 'tmp.wav')
    return Audio('tmp.wav')


In [ ]:
from anticipation import ops
import numpy as np

def get_input_tokens(midi_path, num_sec):
    events = midi_to_events(midi_path)

    # Clip to first N seconds and normalize time to start at 0
    unpadded_events = ops.clip(events, 0, num_sec)
    unpadded_events = ops.translate(unpadded_events, -ops.min_time(unpadded_events, seconds=False))

    # Trim to 510 tokens (aligned to triplets)
    MAX_ALIGNED_EVENTS = ((512 - 1) // 3) * 3
    unpadded_events = unpadded_events[:MAX_ALIGNED_EVENTS]

    # Pad front to 511, prepend AUTOREGRESS to get 512
    padding = [0] * (511 - len(unpadded_events))
    input_tokens = np.array([padding + [AUTOREGRESS] + list(unpadded_events)], dtype=np.int64)
    return input_tokens, unpadded_events


## Helper Functions

In [ ]:
def future_logits_new(logits, curtime):
    """ don't sample events in the past or too far in the future """
    if curtime > 0:
        logits[TIME_OFFSET:TIME_OFFSET+curtime] = -float('inf')

    if curtime < (MAX_TIME - 200):
        logits[TIME_OFFSET+curtime+200:DUR_OFFSET] = -float('inf')

    return logits

In [ ]:
def build_global_attention(unpadded_events, generated_tokens, full_attention_history, head_idx=0):
    # unpadded_events is the unpadded input tokens
    total_content_len = len(unpadded_events) + len(generated_tokens)
    global_attn = np.zeros((total_content_len, total_content_len))
    
    for t, step_attn in enumerate(full_attention_history):
        # window_attn = step_attn[0, head_idx, 0, :] 
        window_attn = step_attn[head_idx, :]
        global_query_idx = len(unpadded_events) + t
        num_real_tokens = min(511, global_query_idx + 1)
        real_attn_slice = window_attn[-num_real_tokens:]
        
        start_col = global_query_idx - num_real_tokens + 1
        global_attn[global_query_idx, start_col : global_query_idx + 1] = real_attn_slice
    return global_attn

In [ ]:
import json
def tokens_to_note_objects(tokens, global_attn, token_offset_in_global, data_type):
    """
    data_type: 'time' (offset 0), 'dur' (offset 1), 'pitch' (offset 2)
    get every 3rd row
    """
    type_offset = {'time': 0, 'dur': 1, 'pitch': 2}[data_type]
    compound = [int(x) for x in events_to_compound(list(tokens))]
    note_objects = []
    for i in range(len(compound) // 5):
        t, d, p, _inst, _v = compound[i*5:i*5+5]
        token_idx = token_offset_in_global + (i * 3) + type_offset
        note_objects.append({
            "pitch": p,
            "startTime": float(t),
            "endTime": float(t+d),
            "attention": global_attn[token_idx].tolist()
        })
    return note_objects

## Run Chick Corea Model on Input Tokens to Generate More Tokens. Output Attention Weights.

In [ ]:
import onnxruntime

MAX_LEN = 512
# midi_path = "/afs/csail.mit.edu/u/r/rjloh/rachel_run_model/anticipation/examples/strawberry.mid"
# midi_path = "/scratch/rjloh/samples/pijama_chick_corea/Oblivion - Live.midi"
# midi_path = "/scratch/rjloh/samples/pijama_keith_jarrett/Pt. III - Salle Pleyel, Paris - Live.midi"
# midi_path = "/scratch/rjloh/samples/jordan/jordan_trading_09.mid"
midi_path = "/scratch/rjloh/samples/unmusical/extreme_register_1.mid"
midi_name = "extreme_register_1"

In [ ]:
events = midi_to_events(midi_path)
input_ids, unpadded_events = get_input_tokens(midi_path, 34)
print(f"unpadded_events length: {len(unpadded_events)}")

In [ ]:
def find_optimal_clip_seconds(midi_path, target_tokens=510, step=1):
    """Find the number of seconds to clip to get closest to target_tokens while being at least target_tokens."""
    events = midi_to_events(midi_path)
    print(f"Total events in full MIDI: {len(events)}")
    
    best_sec = 1
    best_len = 0
    
    sec = 1
    while True:
        clipped = ops.clip(events, 0, sec)
        n = len(clipped)
        
        if n > target_tokens:
            print(f"Optimal: {best_sec}s → {best_len} tokens ({best_len//3} notes)")
            print(f"Next: {sec}s → {n} tokens (exceeds {target_tokens})")
            return sec
        
        best_sec = sec
        best_len = n
        sec += step

# optimal_sec = find_optimal_clip_seconds(midi_path)
input_ids, unpadded_events = get_input_tokens(midi_path, 34)
print(f"unpadded_events length: {len(unpadded_events)}")

## Audio Playback of Input

In [ ]:
synthesize_og(fs, unpadded_events)

## Generate Tokens (60 tokens - half context window)

In [ ]:
import torch.nn.functional as F

def add_token_onnx(session, z, tokens, current_time):
    """add_token() for ONNX inference."""
    assert len(tokens) % 3 == 0
    
    history = tokens.copy()
    lookback = max(len(tokens) - 1017, 0)
    history = history[lookback:]  # Markov window
    offset = ops.min_time(history, seconds=False)
    history[::3] = [tok - offset for tok in history[::3]]  # relativize time
    
    new_token = []
    attention_per_token = []  # stores (16, 511) attention for each of 3 tokens
    
    for i in range(3):
        input_seq = z + history + new_token
        if len(input_seq) > 512:
            input_seq = [z[0]] + input_seq[-511:]
        pad_len = 512 - len(input_seq)
        padded = [0] * pad_len + input_seq
        input_ids = np.array([padded], dtype=np.int64)
        attention_mask = np.concatenate((np.zeros(pad_len), np.ones(input_seq)))
        
        ort_outputs = session.run(None, {"input_ids": input_ids, "attention_mask": attention_mask})
        logits = torch.from_numpy(ort_outputs[0])[0, -1, :]
        
        # Store attention for all heads, skip AUTOREGRESS col
        attn = ort_outputs[-1][0, :, -1, 1:]  # shape (16, 511)
        attention_per_token.append(attn)
        
        logits = safe_logits(logits, i)
        if i == 0:
            logits = future_logits_new(logits, current_time - offset)
            logits[REST] = -float('inf')  # mask out REST token
        elif i == 2:
            logits = instr_logits(logits, tokens)
        # logits = future_logits_new(logits, current_time - offset)

        # next_token = int(torch.argmax(logits))
        # new_token.append(next_token)
        logits = nucleus(logits, top_p=1.0)
        probs = F.softmax(logits, dim=-1)
        next_token = int(torch.multinomial(probs, 1))
        new_token.append(next_token)
    
    new_token[0] += offset  # revert to full sequence timing
    return new_token, attention_per_token  # attention still in relativized time space, that's fine


# Generation loop
tokens = list(unpadded_events)
current_time = max(unpadded_events[::3])
generated_tokens = []
session = onnxruntime.InferenceSession(onnx_model_path, providers=["CPUExecutionProvider"])
NUM_NOTES = 60

full_attention_history = []

for _ in range(NUM_NOTES):
    new_token, attention_per_token = add_token_onnx(session, [AUTOREGRESS], tokens, current_time)
    tokens.extend(new_token)
    generated_tokens.extend(new_token)
    current_time = max(current_time, new_token[0])
    full_attention_history.extend(attention_per_token)

## Audio Playback of Generated Tokens
### start playback where the generation actually begins

In [ ]:
normalized = generated_tokens.copy()
first_time = normalized[0]
normalized[::3] = [t - first_time for t in normalized[::3]]

synthesize_og(fs, normalized)

## Reformat Attention to be Compatible With MusicViz
### Make Attention Into Global Triangular Matrix
### Also Separate Out Pitch, Duration, and Time Tokens so They Each Get an Attention Matrix

In [ ]:
import zipfile
import os

NUM_HEADS = 16
base_path = "/scratch/rjloh/attention_weights"
folder_name = os.path.join(base_path, midi_name)
os.makedirs(folder_name, exist_ok=True)
zip_filename = f"{folder_name}.zip"

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for head_idx in range(NUM_HEADS):
        global_matrix = build_global_attention(unpadded_events, generated_tokens, full_attention_history, head_idx)
        
        for data_type in ['time', 'dur', 'pitch']:
            prompt_notes = tokens_to_note_objects(
                unpadded_events, global_matrix, 0, data_type
            )
            for note in prompt_notes:
                note["attention"] = [0.0] * global_matrix.shape[1]
            
            gen_notes = tokens_to_note_objects(
                generated_tokens, global_matrix, len(unpadded_events), data_type
            )
            
            full_note_objects = prompt_notes + gen_notes
            filename = f"{midi_name}{head_idx}_{data_type}.ts"
            filepath = os.path.join(folder_name, filename)
            ts_content = f"export const head{head_idx}_{data_type} = {json.dumps(full_note_objects, indent=2)};"
            with open(filepath, "w") as f:
                f.write(ts_content)
            zipf.write(filepath, arcname=os.path.join(midi_name, filename))
            print(f"Exported {filepath}: {len(prompt_notes)} prompt + {len(gen_notes)} generated")

print(f"All files zipped into {zip_filename}")

## Break Jordan Data Into 10 Samples of ~510 Tokens Each

In [ ]:
midi_path = "/scratch/rjloh/samples/jordan_trading.mid"

In [ ]:
def split_midi_into_segments(midi_path, num_segments=10, target_tokens=510, step=1):
    events = midi_to_events(midi_path)
    total_duration = ops.max_time(events, seconds=True)
    print(f"Total events: {len(events)}, Total duration: {total_duration:.1f}s")
    
    segments = []
    seg_start = 0  # in seconds
    
    while len(segments) < num_segments:
        # Clip events from seg_start onwards and translate to start at 0
        remaining = ops.clip(events, seg_start, total_duration)
        remaining = ops.translate(remaining, -ops.min_time(remaining, seconds=False))
        
        if len(remaining) < target_tokens:
            print(f"Not enough events remaining for segment {len(segments)+1}, stopping.")
            break
        
        # Find smallest number of seconds >= target_tokens
        sec = step
        while True:
            clipped = ops.clip(remaining, 0, sec)
            n = len(clipped)
            if n >= target_tokens:
                # Align to multiple of 3
                aligned = (n // 3) * 3
                clipped = clipped[:aligned]
                break
            sec += step
        
        # Save segment as midi
        seg_midi = events_to_midi(clipped)
        seg_path = f"/scratch/rjloh/samples/jordan/jordan_trading_{len(segments):02d}.mid"
        os.makedirs("/scratch/rjloh/samples/jordan", exist_ok=True)
        seg_midi.save(seg_path)
        
        # Advance start time by the duration of this segment
        seg_duration = ops.max_time(clipped, seconds=True)
        print(f"Segment {len(segments):02d}: start={seg_start:.1f}s, duration={seg_duration:.1f}s, tokens={len(clipped)}")
        
        segments.append(clipped)
        seg_start += seg_duration
    
    print(f"Saved {len(segments)} segments.")
    return segments

segments = split_midi_into_segments(midi_path, num_segments=10)

## Create 10 OOD Random "Unmusical" Samples

In [ ]:
import random
from anticipation.convert import events_to_midi

def make_events(time_dur_note_triples):
    """Convert list of (time, dur, note) to flat event list with offsets."""
    events = []
    for t, d, n in time_dur_note_triples:
        events.extend([t, d + DUR_OFFSET, n + NOTE_OFFSET])
    return events

def ensure_510(triples, repeat_fn):
    """Keep generating until we have >= 510 tokens."""
    while len(triples) * 3 < 510:
        triples.extend(repeat_fn())
    return triples[:((len(triples) * 3 // 3) * 3 // 3)]  # align to note boundary

def save_sample(events, path):
    mid = events_to_midi(events)
    mid.save(path)
    print(f"Saved {path} ({len(events)} tokens = {len(events)//3} notes)")

os.makedirs("/scratch/rjloh/samples/unmusical", exist_ok=True)

# 1. Smashing the piano — dense clusters of all notes at once
def piano_smash():
    triples = []
    t = 0
    while len(triples) * 3 < 510:
        # Play 10-20 random notes all at the same time
        cluster_size = random.randint(10, 20)
        for _ in range(cluster_size):
            pitch = random.randint(0, 127)
            dur = random.randint(10, 50)
            triples.append((t, dur, pitch))
        t += random.randint(5, 30)
    return triples

for i in range(2):
    triples = piano_smash()
    events = make_events(triples)
    save_sample(events, f"/scratch/rjloh/samples/unmusical/smash_{i}.mid")

# 2. Random durations and random note lengths
def random_everything():
    triples = []
    t = 0
    while len(triples) * 3 < 510:
        pitch = random.randint(0, 127)
        dur = random.randint(1, MAX_DUR - 1)
        time_gap = random.randint(0, 500)
        triples.append((t, dur, pitch))
        t += time_gap
    return triples

for i in range(2):
    triples = random_everything()
    events = make_events(triples)
    save_sample(events, f"/scratch/rjloh/samples/unmusical/random_{i}.mid")

# 3. Repeat 2 notes over and over
def two_note_loop():
    triples = []
    t = 0
    note_a = random.randint(0, 127)
    note_b = random.randint(0, 127)
    dur = random.randint(10, 100)
    gap = random.randint(10, 50)
    while len(triples) * 3 < 510:
        triples.append((t, dur, note_a))
        t += gap
        triples.append((t, dur, note_b))
        t += gap
    return triples

for i in range(2):
    triples = two_note_loop()
    events = make_events(triples)
    save_sample(events, f"/scratch/rjloh/samples/unmusical/two_note_{i}.mid")

# 4. Single note over and over
def single_note_loop():
    triples = []
    t = 0
    pitch = random.randint(0, 127)
    dur = random.randint(10, 100)
    gap = random.randint(10, 50)
    while len(triples) * 3 < 510:
        triples.append((t, dur, pitch))
        t += gap
    return triples

for i in range(2):
    triples = single_note_loop()
    events = make_events(triples)
    save_sample(events, f"/scratch/rjloh/samples/unmusical/single_note_{i}.mid")

# 5. Extreme registers only
def extreme_registers():
    triples = []
    t = 0
    use_high = random.choice([True, False])
    dur = random.randint(10, 100)
    gap = random.randint(10, 50)
    while len(triples) * 3 < 510:
        if use_high:
            pitch = random.randint(100, 127)
        else:
            pitch = random.randint(0, 20)
        triples.append((t, dur, pitch))
        t += gap
        use_high = not use_high  # alternate between high and low
    return triples

for i in range(2):
    triples = extreme_registers()
    events = make_events(triples)
    save_sample(events, f"/scratch/rjloh/samples/unmusical/extreme_register_{i}.mid")

print("Done! All 10 unmusical samples saved.")

## Debugging (IGNORE)

In [ ]:
print(f"Last prompt time: {max(unpadded_events[::3])}")
print(f"Generated time tokens (raw): {generated_tokens[::3]}")
print(f"Generated time tokens (normalized): {normalized[::3]}")
print(f"Time gaps between notes: {[normalized[::3][i+1] - normalized[::3][i] for i in range(len(normalized[::3])-1)]}")

In [ ]:
# Find how many notes are in the prompt
prompt_notes = sum(1 for m in token_map[:len(prompt_content)] if m is not None and m[1] == 'pitch')
generated_notes = len(global_compound_data) // 5

print(f"Prompt notes: {prompt_notes}")
print(f"Generated notes: {generated_notes}")

# Now re-check with correct boundary
for i in range(min(3, len(head_attention))):
    note = head_attention[i]
    attn = np.array(note["attention"])
    print(f"Note {i} pitch={note['pitch']}")
    print(f"  Full vector sum: {attn.sum():.4f}, length: {len(attn)}")
    print(f"  Prompt region [0:{prompt_notes}] sum: {attn[:prompt_notes].sum():.4f}")
    print(f"  Generated region [{prompt_notes}:] sum: {attn[prompt_notes:].sum():.4f}")
    print(f"  Generated region values (nonzero): {attn[prompt_notes:][attn[prompt_notes:] > 0]}")

In [ ]:
prompt_notes = 170

for i in [0, 15, 25]:
    note = head_attention[i]
    attn = np.array(note["attention"])
    print(f"Note {i} pitch={note['pitch']}, attn length={len(attn)}")
    print(f"  Prompt region [0:170] sum: {attn[:prompt_notes].sum():.4f}")
    print(f"  Generated region [170:] sum: {attn[prompt_notes:].sum():.4f}")
    print(f"  Generated region (nonzero): {attn[prompt_notes:][attn[prompt_notes:] > 0]}")

In [ ]:
print(f"global_compound_data first 15: {global_compound_data[:15]}")
print(f"prompt_compound first 15: {prompt_compound[:15]}")

In [ ]:
# Use the same stripping logic as safe_convert_to_midi
prompt_events = [t for t in prompt_content if t > 0 and t != ANTICIPATE]

# Find phase alignment
start_idx = -1
for i in range(min(10, len(prompt_events))):
    if prompt_events[i] < 1000:
        start_idx = i
        break

print(f"start_idx: {start_idx}")
print(f"first few prompt_events: {prompt_events[:10]}")

In [ ]:
from anticipation.vocab import *
print(TIME_OFFSET if 'TIME_OFFSET' in dir() else "not found")
print(NOTE_OFFSET if 'NOTE_OFFSET' in dir() else "not found")

In [ ]:
import anticipation.vocab as vocab
print(dir(vocab))

In [ ]:
events = midi_to_events(midi_path)
print(f"First 15 raw events: {events[:15]}")
print(f"Min event value: {min(events)}")
print(f"Max event value: {max(events)}")

In [ ]:
print(f"TIME_OFFSET: {TIME_OFFSET}")
print(f"ATIME_OFFSET: {ATIME_OFFSET}")
print(f"DUR_OFFSET: {DUR_OFFSET}")
print(f"ADUR_OFFSET: {ADUR_OFFSET}")
print(f"NOTE_OFFSET: {NOTE_OFFSET}")
print(f"ANOTE_OFFSET: {ANOTE_OFFSET}")

# See if subtracting ATIME_OFFSET from prompt time tokens gives sensible values
print(f"\nFirst prompt token minus ATIME_OFFSET: {prompt_content[0] - ATIME_OFFSET}")
print(f"First raw event time: {events[0]}")

In [ ]:
print(f"Last few unpadded_events time tokens: {unpadded_events[::3][-5:]}")
print(f"First few generated time tokens: {generated_tokens[::3][:5]}")